In [ ]:
import pandas as pd
from utils.utils_feature_extraction import *
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*empty or all-NA entries.*")

### Performing feature extraction from pipeline outputs
After running the pipeline, you can use this script to extract features describing lesion shape and distribution from the pipeline outputs. We provide two ways to do this. First is a _per lesion_ feature extraction using the individual lesions identified from the 3D connected components analysis. We use this to define affected vascular territories in our paper. This code assigns each individual lesion to the atlas territory it shares greatest overlap with. It also calcuates shape features per lesion as defined in pyradiomics details on the definition of the shape features and their units can be found in the pyradiomics documentation or our paper. It returns a pandas dataframe where each row corresponds to one individual lesion, with  information on the vascular territory assingment, the percentage overlap with this territory, lesion volume and coordinates of the centroid of the lesion in MNI space. This pandas dataframe can be aggregated to patient level feautes using the aggregate_features.ipynb Notebook. 

By default, lesion volumes are returned in ml, except for MeshVolume, which is returrned in mm3.

Second, there is a _global_ feature extraction. Here the atlas is overlayed with the registered lesion mask and lesioned volume per atlas territory is directly calculated without assigning each lesion to one territory. It returns a pandas dataframe where each row corresponds to one patient and each column to one atlas territory. We use this code to define patients with cortical lesions in our paper. 

You can either use the atlases we use in our paper or specify your own atlases in FSL MNI 152 space for both global and per lesion feature extraction

### Specify paths

In [ ]:
patients = pd.read_csv("path/to/masterfile.csv")
preprocessed_path = "path/to/pipeline/outputs"

per_lesion_atlas_path = "path/to/vascular/atlas.nii.gz" #Used for per lesion feature extraction
per_lesion_atlas_labels = "path/to/labels/file.txt" # should contain mapping from atlas labels to names of territories

global_atlas_path = "path/to/anatomical/atlas.nii.gz" #Used for global feature extraction
global_atlas_labels_path = "path/to/label/file.txt"

### Per lesion feature extraction (e.g. vascular territories)

In [ ]:
atlas = nib.load(per_lesion_atlas_path)
atlas = atlas.get_fdata()[:,:,:,0]
atlas = np.round(atlas).astype(np.uint16)
lesion_distribution = None

for index, row in patients.iterrows():
    sub_id = row["Patient ID"]
    if "Center" in patients.columns:
        center = row["Center"]
        path = os.path.join(preprocessed_path, center)
    else: 
        path = preprocessed_path
        center = np.nan

    try:
        img_nat, vox_dim, shape = load_image(sub_id, path,  "_lesion_components.nii.gz", mask = True)
        total_volume = calculate_total_lesion_volume(img_nat, vox_dim, unit="ml")
        img_mni, _, _ = load_image(sub_id, path, "_lesion_components_mni.nii.gz", mask = True)
        
        lesion_info, n_lesions = analyze_stroke_lesions(img_nat, img_mni, vox_dim, atlas, excluded_territories={0,9,10}, unit = "ml")
        shapes = extract_lesion_shape_features(
            lesion_mask=img_nat,
            spacing=vox_dim
        )


        combined = pd.concat(
            [lesion_info.reset_index(drop=True), shapes.reset_index(drop=True)],
            axis=1
        )

        combined.insert(0, 'Patient ID', sub_id)
        combined.insert(2, 'Center', center)       
        combined['Total Lesion Volume'] = total_volume 
        combined['N Lesions'] = n_lesions 

        if lesion_distribution is None:
            lesion_distribution = combined
        else:
            lesion_distribution = pd.concat([lesion_distribution.reset_index(drop=True), combined.reset_index(drop=True)], axis=0)

    except FileNotFoundError:
        if lesion_distribution is not None:
            na_row = pd.DataFrame([{col: np.nan for col in lesion_distribution.columns}])
            na_row['Patient ID'] = sub_id
            na_row['Center'] = center
            lesion_distribution = pd.concat([lesion_distribution, na_row], ignore_index=True)
        else:
            cols = ['Patient ID', 'Lesion ID', 'Center', 'Individual lesion volume', 'centroid',
                    'region', 'overlap percentage', 'MeshVolume', 'SurfaceArea',
                    'SurfaceVolumeRatio', 'Sphericity', 'Maximum3DDiameter',
                    'Maximum2DDiameterSlice', 'Maximum2DDiameterColumn',
                    'Maximum2DDiameterRow', 'MajorAxisLength', 'MinorAxisLength',
                    'LeastAxisLength', 'Elongation', 'Flatness', 'N voxels',
                    'Total Lesion Volume', 'N Lesions']
            na_row = pd.DataFrame([{col: np.nan for col in cols}])
            na_row['Patient ID'] = sub_id
            na_row['Center'] = center
            lesion_distribution = na_row


### (Optional) Replace labels with label names
Here you can replace the labels with the label names. The load_label_dict_txt() function is specific for the label file for the vascular territory atlas we use, with the level of the atlas (1 or 2) to be specified. If you are using a different atlas, you will can either write your own loading function or specify the mapping dict directly. The mapping dict has to be a dictionary with labels as keys and label names as values.

In [ ]:
mapping_dict = load_label_dict_txt(per_lesion_atlas_labels, level=2)
lesion_distribution = pd.DataFrame(lesion_distribution)
lesion_distribution["region"] = lesion_distribution["region"].map(mapping_dict)

In [ ]:
lesion_distribution.to_csv("/path/to/save.csv")

### Global feature extraction (e.g. cortical regions)

In [ ]:
atlas = nib.load(global_atlas_path)
atlas = atlas.get_fdata()
labels = np.unique(atlas)
global_distribution = None
for index, row in patients.iterrows():
    sub_id = row["Patient ID"]
    if "Center" in patients.columns:
        center = row["Center"]
        path = os.path.join(preprocessed_path, center)
    else: 
        path = preprocessed_path
        center = np.nan

    try:
        img_mni, vox_dim, _ = load_image(sub_id, path, "_lesion_components_mni.nii.gz", mask = True)
        df = lesion_atlas_overlap(img_mni, atlas, vox_dim)
    except FileNotFoundError:
        df = pd.DataFrame([{col: np.nan for col in labels}])

    df.insert(0, "Patient ID", [sub_id])
    df.insert(1, "Center", [center])
    if global_distribution is None:
        global_distribution = df
    else:
        global_distribution = pd.concat([global_distribution, df], axis=0)
        

### (Optional) Replace labels with label names
Here you can replace the labels with the label names. The load_label_dict_xml() function is specific for the label file for the anatomical atlas we use. If you are using a different atlas, you will can either write your own loading function or specify the mapping dict directly. The mapping dict has to be a dictionary with labels as keys and label names as values.

In [ ]:
mapping_dict = load_label_dict_xml(global_atlas_labels_path)
global_distribution = global_distribution.rename(mapper=mapping_dict, axis=1)

In [ ]:
global_distribution.to_csv("/path/to/save.csv")